# 03. 로컬 LLM 서버 구축 — llama.cpp `llama-server` (Qwen3.5-4B / Qwen2.5-7B)

**이 노트북에서 띄운 서버를 05·07 핵심 실습과 06·08·09 심화 실습에서 공용으로 사용합니다.**
끝까지 실행한 뒤 **서버를 끄지 마세요.**

Windows의 llama-server 실행 파일을 로컬 프로세스로 시작합니다.
모델은 **프리셋 2개 중 하나**를 고릅니다 (2번 셀의 `LLM_PRESET` / 정의는 `local_llm.py`의 `LLM_PRESETS`).

| 프리셋 | 모델 | 파일 크기 | VRAM(ctx 16k) | 특징 |
|---|---|---|---|---|
| `'4b'` **(기본)** | Qwen3.5-4B GGUF · `Q8_0` + `mmproj-F16` | ≈4.5 GB (+0.7) | ≈6 GB | 최신 세대, **비전 지원**, 한국어/툴 호출 양호 |
| `'7b'` | `bartowski/Qwen2.5-7B-Instruct-GGUF` · `Q4_K_M` | ≈4.7 GB | ≈6.5 GB | 더 큰 파라미터, **텍스트 전용** |

- 컨텍스트 16,384 토큰 (에이전트의 tool 스키마/대화 이력을 넉넉히 수용), 응답 최대 4,096 토큰
- 엔드포인트 `http://localhost:8080/v1` (OpenAI 호환, API 키 불필요)

> Qwen3.5에는 7B 크기가 없어서(0.8B/2B/4B/9B), 7B 옵션은 llama.cpp tool calling이 검증된 **Qwen2.5-7B-Instruct**를 사용합니다.
> 7B 프리셋은 비전 프로젝터가 없으므로 6번(비전 테스트) 셀은 자동으로 건너뜁니다.
> Qwen3.5 소형 모델은 기본적으로 thinking(추론 모드)이 꺼져 있으며, 서버에서도 명시적으로 끕니다. (Qwen2.5는 thinking 개념이 없어 해당 옵션은 무시됨)

**소요 시간:** 최초 실행 시 ~5-10분 (대부분 이미지/모델 다운로드). 이후 재실행 ~1분.

**데모 절차:**
Windows의 llama-server 실행 파일을 로컬 프로세스로 시작합니다.
2. `ghcr.io/ggml-org/llama.cpp:server-cuda` 이미지가 pull 및 실행 가능한지
3. GGUF (+ 4B는 `mmproj-F16.gguf`)가 A4000에 로딩되는지
4. OpenAI 호환 API가 텍스트 전용 채팅에 응답하는지
5. 동일 API가 비전 입력 (`image_url`)에 올바르게 응답하는지 (4B 프리셋만)
6. 도구 호출(raw API / LangChain)이 동작하는지
7. **05~09 실습이 쓰는 기능 전체 점검** (json_schema 구조화 출력 / ToolStrategy 에이전트 / 다중 tool 루프) → 모두 통과해야 다음 실습 진행

**사전 요구 사항 (다음 셀에서 확인, 자동 설치 아님):**
- Windows에서 `setup_environment.bat` 설치 완료
- NVIDIA 드라이버와 CUDA 지원 PyTorch 확인 완료
- Python 의존성: `huggingface_hub`, `openai`, `Pillow`, `requests`, `langchain`, `langchain-openai`
- 이미지 + 모델 아티팩트용 여유 디스크 공간 ~10 GB

> 모델은 Hugging Face 공개 저장소에서 받습니다. 실습에서는 GGUF 모델 파일만 사용하며 Unsloth 학습 라이브러리는 사용하지 않습니다.


## 🧭 시작 전에

> **30분 핵심 경로**: 기본 4B 모델 선택 → 서버 시작·상태 확인 → 텍스트 생성 → 도구 호출 1회. 비전 테스트와 전체 기능 점검은 선택입니다.

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | Windows에서 로컬 LLM 서버를 실행하고 텍스트·이미지·도구 호출을 확인해 05~09 실습을 준비한다 |
| 📚 사전 지식 | 00번(환경 확인), 01번(Transformers) |
| 🐍 커널 | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | llama-server가 약 6~6.5 GB 사용 |
| ▶ 실행 | 모델 다운로드 후 서버 시작 셀을 실행하고, 05~09가 끝나면 서버를 종료한다 |

위에서부터 셀을 순서대로 실행하세요. `Run All`은 사용하지 마세요.


## What this notebook starts

The bundled Windows llama.cpp executable loads the selected GGUF model directly on the RTX A4000 and serves an OpenAI-compatible API at `http://127.0.0.1:8080/v1`. 서버는 Windows 로컬 프로세스로 실행됩니다.

Use `local_server.start_server` to start it and `local_server.stop_server(8080)` after the 05-09 labs.

### 설치 확인

필요한 라이브러리와 Windows llama-server는 `setup_environment.bat`이 설치합니다. 패키지 오류가 나면 프로젝트 폴더에서 설치기를 다시 실행한 뒤 이 커널을 재시작하세요.


## 0. 사전 요구 사항 점검

현재 커널에서 CUDA GPU와 설치된 `llama-server.exe`를 확인합니다.


In [1]:
import torch, shutil
from pathlib import Path
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CUDA unavailable')
assert Path('bin/llama-server.exe').exists(), 'Run setup_environment.bat first'
print('Bundled llama-server executable ready')

GPU: NVIDIA RTX A4000
Bundled llama-server executable ready


설치기는 Windows 실행 파일과 CUDA 런타임 DLL을 `bin/`에 준비합니다.


## 1. 로컬 서버 파일 확인

추가 서버 파일 다운로드 없이 설치된 Windows 실행 파일을 확인합니다.


In [2]:
from pathlib import Path
assert Path('bin/llama-server.exe').exists(), 'Run setup_environment.bat first'
print('No image download needed; using bin/llama-server.exe')

No image download needed; using bin/llama-server.exe


## 2. 모델 선택 + GGUF 다운로드

`LLM_PRESET`을 `'4b'`(기본) 또는 `'7b'`로 정하고 실행하세요.

- `'4b'`: Qwen3.5-4B `Q8_0` (≈4.5 GB) + 멀티모달 프로젝터 `mmproj-F16`
- `'7b'`: Qwen2.5-7B-Instruct `Q4_K_M` (≈4.7 GB), 텍스트 전용
- 저장 위치: `~/models/<저장소 이름>` (서버 실행과 노트북이 같은 파일을 사용합니다)
- 4B는 8bit(Q8_0), 7B는 4bit(Q4_K_M) — 04번 KLD 실습에서 두 양자화의 품질 차이가 얼마나 작은지 직접 확인합니다.


In [3]:
import os

# ── 모델 선택 (A4000 16GB 기준) ──
LLM_PRESET = '4b'   # '4b' (기본, Qwen3.5-4B Q8_0, 비전 지원) 또는 '7b' (Qwen2.5-7B-Instruct Q4_K_M, 텍스트 전용)

os.environ['LLM_PRESET'] = LLM_PRESET   # local_llm이 같은 프리셋을 쓰도록 import 전에 설정
from local_llm import LLM_PRESETS, preset_model_dir, find_gguf_files
P = LLM_PRESETS[LLM_PRESET]
LLM_REPO, LLM_QUANT, HAS_VISION = P['repo'], P['quant'], P['vision']
print(f"선택: {LLM_PRESET} → {P['desc']}")

MODEL_DIR = preset_model_dir(LLM_PRESET)   # 프로젝트 models 폴더 (셋업에서 사전 다운로드)
os.makedirs(MODEL_DIR, exist_ok=True)

# 셋업이 모델을 미리 받습니다. 누락된 경우 setup_environment.bat을 다시 실행하세요.
# 실제 다운로드된 파일 탐색
MODEL_PATH, MMPROJ_PATH = find_gguf_files(MODEL_DIR, LLM_QUANT, need_mmproj=HAS_VISION)


print(f'Model:   {MODEL_PATH}  ({os.path.getsize(MODEL_PATH) / 1024**3:.2f} GB)')
if MMPROJ_PATH:
    print(f'MMProj:  {MMPROJ_PATH}  ({os.path.getsize(MMPROJ_PATH) / 1024**3:.2f} GB)')
else:
    print('MMProj:  (없음 — 텍스트 전용 모델)')
print()
print('모델 파일과 로컬 서버 실행 준비 완료')

# 다음 셀에서 사용할 모델 파일 경로 저장
os.environ['MODEL_DIR'] = MODEL_DIR


선택: 4b → Qwen3.5-4B Q8_0 (≈4.5GB, 비전 지원) — 기본
Model:   d:\유연휘\LLM_Research_Lab_Student\models\Qwen3.5-4B-GGUF\Qwen3.5-4B-Q8_0.gguf  (4.17 GB)
MMProj:  d:\유연휘\LLM_Research_Lab_Student\models\Qwen3.5-4B-GGUF\mmproj-F16.gguf  (0.63 GB)

모델 파일과 로컬 서버 실행 준비 완료


## 3. llama-server 시작

다음 셀은 `bin/llama-server.exe`를 Windows 로컬 프로세스로 시작하고 서버 준비가 끝나면 반환합니다.


`local_server.start_server()`가 Windows 실행 파일을 직접 시작하며, 주요 설정은 다음과 같습니다.

- `--n-gpu-layers 99`: 모델 레이어를 GPU에 최대한 올림
- `--ctx-size 16384`: 대화와 도구 스키마에 사용할 문맥 길이
- `--parallel 1`: 실습 요청을 한 번에 하나씩 처리
- `--n-predict 4096`: 응답 길이 상한
- `--jinja`: GGUF 채팅 템플릿과 도구 호출 활성화
- `--mmproj`: 4B 비전 모델의 이미지 프로젝터 (7B 텍스트 모델에서는 생략)


In [4]:
from local_server import start_server
SERVER_PID = start_server(
    MODEL_PATH, port=8080, mmproj_path=MMPROJ_PATH or None,
    ctx_size=16384, n_predict=4096, jinja=True, timeout=300,
)
print('PID:', SERVER_PID)

llama-server already responds on port 8080
PID: 27200


## 4. 서버 상태 확인

앞 셀의 `start_server()`가 `/health` 준비 상태까지 기다렸습니다. 여기서는 API 상태를 한 번 더 확인합니다.


In [5]:
import requests

response = requests.get('http://127.0.0.1:8080/health', timeout=5)
response.raise_for_status()
print(response.json())


{'status': 'ok'}


**서버 로그 보는 법** — 문제가 생기면 가장 먼저 로그를 봅니다. 아래 셀은 서버 로그의 마지막 부분을 보여 줍니다.
`load_tensors`, `offloaded 99/99 layers to GPU`(또는 비슷한 문구), `server is listening` 같은 줄이 보이면 정상입니다.

In [6]:
from pathlib import Path
log = Path('runtime/llama-server-8080.log')
print(log.read_text(encoding='utf-8', errors='replace')[-4000:] if log.exists() else 'No server log yet')

0.00.151.678 W Setting 'enable_thinking' via --chat-template-kwargs is deprecated. Use --reasoning on / --reasoning off instead.
0.00.151.891 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.152.487 W srv  llama_server: -----------------
0.00.152.496 W srv  llama_server: CORS is set to allow all origins ('*') and no API key is set
0.00.152.498 W srv  llama_server: this can be a security risk (cross-origin attacks)
0.00.152.499 W srv  llama_server: more info: https://github.com/ggml-org/llama.cpp/pull/25655
0.00.152.499 W srv  llama_server: -----------------
0.00.156.110 I srv    load_model: loading model 'D:\유연휘\LLM_Research_Lab_Student\models\Qwen3.5-4B-GGUF\Qwen3.5-4B-Q8_0.gguf'
0.04.325.773 I cmn          init: llama threadpool init, n_threads = 8
0.04.428.675 W load_hparams: Qwen-VL models require at minimum 1024 image tokens to function correctly on grounding tasks
0.04.428.683 W load_hparams: if you encounter problems with accura

## 5. 텍스트 테스트 — OpenAI 호환 API

llama-server는 표준 `/v1/chat/completions` 형식을 지원합니다. `model` 필드는 무시되지만 (로드된 모델은 `-m` 플래그로 결정) 로그 가독성을 위해 라벨을 전달합니다.

### 📖 OpenAI 호환 API 요청의 구조

```python
client.chat.completions.create(
    model='...',                       # llama-server는 무시 (이미 로드된 모델 사용)
    messages=[                         # 대화 기록 (역할 + 내용)
        {'role': 'system',    'content': '당신은 ...'},     # 규칙/역할 지정 (선택)
        {'role': 'user',      'content': '질문'},           # 사용자 입력
        {'role': 'assistant', 'content': '이전 답변'},      # 멀티턴일 때 이전 답
    ],
    temperature=0,                     # 01번에서 배운 디코딩 설정
    max_tokens=32,                     # 생성할 최대 토큰 수
)
# 응답: resp.choices[0].message.content (답변 텍스트), resp.usage (사용한 토큰 수)
```

> 모델은 **이전 대화를 스스로 기억하지 않습니다.** 멀티턴 대화는 매 요청마다 이전 메시지를 `messages`에 다시 넣어 보내는 방식입니다. (06번의 메모리 기능이 이 일을 자동으로 해 줍니다)

In [9]:
from openai import OpenAI
from local_llm import LLM_BASE_URL, LLM_API_KEY, LLM_MODEL   # 공용 설정 (localhost:8080)

client = OpenAI(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,  # llama-server는 인증 불필요 (아무 문자열)
)

resp = client.chat.completions.create(
    model=LLM_MODEL,      # llama-server는 model 필드를 무시 (로그용 라벨)
    messages=[
        {'role': 'user', 'content': '너는 누구니'},
    ],
    max_tokens=1000,
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print('응답:', repr(resp.choices[0].message.content))
print('토큰 수:', resp.usage.completion_tokens)


응답: '안녕하세요! 저는 **Qwen3.5**입니다. 알리바바 클라우드에서 개발한 초대규모 언어 모델입니다.\n\n저는 다양한 분야에서 도움을 드릴 수 있도록 설계되었습니다. 예를 들어:\n\n*   **복잡한 논리 및 수학 문제 해결**\n*   **코드 생성, 디버깅 및 설명**\n*   **장문 문서 요약 및 분석**\n*   **다국어 번역 및 대화**\n*   **창의적인 글쓰기 및 아이디어 제안**\n\n어떤 도움이 필요하신가요? 궁금한 점이 있거나 작업을 도와드릴까요?'
토큰 수: 126


## 선택 실습: 비전 테스트 (4B 프리셋만 — 7B는 자동 생략)

결정적 테스트 이미지(흰색 배경에 빨간 원)를 생성하고, base64로 인코딩하여 표준 OpenAI `image_url` 컨텐츠 블록으로 전송합니다. 모델은 **빨간색(red)** 과 **원(circle)** (또는 동등한 표현)을 언급해야 합니다.

이것이 작동하면 `--mmproj` 플래그가 비전 프로젝터를 올바르게 연결하고 있으며, 멀티모달 스택이 엔드투엔드로 기능하는 것입니다.

In [10]:
# 7B 프리셋(텍스트 전용)은 mmproj가 없으므로 비전 테스트를 건너뜁니다.
if HAS_VISION:
    from PIL import Image, ImageDraw
    import base64, io

    img = Image.new('RGB', (256, 256), 'white')
    ImageDraw.Draw(img).ellipse((48, 48, 208, 208), fill='red')

    buf = io.BytesIO()
    img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image_url',
                 'image_url': {'url': f'data:image/png;base64,{b64}'}},
                {'type': 'text', 'text': 'What color and shape do you see in this image? Answer in one short sentence.'},
            ],
        }],
        max_tokens=200,
        temperature=0,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},

    )

    answer = resp.choices[0].message.content
    print('응답:', answer)
    print()

    lower = answer.lower()
    saw_red = any(w in lower for w in ('red', 'crimson', 'scarlet', '빨간', '빨갛', '적색'))
    saw_circle = any(w in lower for w in ('circle', 'circular', 'round', 'disk', 'disc', 'ellipse', '원', '동그라미'))

    print(f'빨간색 계열 언급 여부:  {saw_red}')
    print(f'원형 계열 언급 여부:    {saw_circle}')

    if saw_red and saw_circle:
        print('\n비전 파이프라인 정상 작동')
    else:
        print('\n예상 키워드가 포함되지 않았습니다. 수동 점검이 필요합니다.')
else:
    print('텍스트 전용 프리셋(7b) → 비전 테스트 생략')


응답: red circle

빨간색 계열 언급 여부:  True
원형 계열 언급 여부:    True

비전 파이프라인 정상 작동


## 7. 도구 호출 테스트 (raw OpenAI 클라이언트)

가짜 도구 하나를 정의하고, 해당 도구가 필요한 질문을 보내어 세 가지를 검증합니다:
1. 모델이 `finish_reason='tool_calls'`를 반환하는지 (환각 답변이 아닌)
2. `tool_calls[0].function.name`이 선언한 도구와 일치하는지
3. `tool_calls[0].function.arguments`가 파싱 가능한 JSON이며 예상 키를 포함하는지

여기서 통과하면 이 스택(Qwen + llama-server --jinja)에서 도구 호출이 작동하는 것입니다.

### 📖 도구 호출(Tool Calling)이란?

LLM은 날씨 조회나 DB 검색을 **직접 실행할 수 없습니다.** 대신 "이 함수를 이 인자로 불러 주세요"라는 **요청(JSON)** 을 돌려줍니다.

```
① 우리 → 모델: 질문 + 사용 가능한 도구 목록(이름·설명·인자 스키마)
② 모델 → 우리: finish_reason='tool_calls', get_current_weather({"location": "Seoul"})
③ 우리 코드가 실제 함수 실행 → 결과 "14°C, 맑음"
④ 우리 → 모델: 이전 대화 + 도구 결과
⑤ 모델 → 우리: "서울은 현재 14°C로 맑습니다."
```

- 모델은 도구의 **설명(description)과 인자 타입**만 보고 언제 어떤 도구를 쓸지 판단합니다 → 설명을 명확히 쓰는 것이 중요합니다.
- 아래 셀은 ①~②만, 다음 8번 셀(LangChain)은 ①~⑤ 전체 루프를 자동으로 수행합니다.

In [9]:
import json

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_current_weather',
        'description': 'Get the current weather in a given location.',
        'parameters': {
            'type': 'object',
            'properties': {
                'location': {
                    'type': 'string',
                    'description': 'City name, e.g. Seoul or San Francisco',
                },
            },
            'required': ['location'],
        },
    },
}]

resp = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {'role': 'user', 'content': "What's the weather in Seoul right now?"},
    ],
    tools=tools,
    tool_choice='auto',
    temperature=0,
    max_tokens=200,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

choice = resp.choices[0]
print(f'finish_reason: {choice.finish_reason}')
print(f'content: {choice.message.content!r}')
print(f'tool_calls: {choice.message.tool_calls}')

assert choice.finish_reason == 'tool_calls', f'Expected tool_calls, got {choice.finish_reason}'
assert choice.message.tool_calls, 'No tool_calls returned'

tc = choice.message.tool_calls[0]
assert tc.function.name == 'get_current_weather', f'Wrong tool: {tc.function.name}'

args = json.loads(tc.function.arguments)  # raises if malformed
assert 'location' in args, f'Missing location arg: {args}'
assert 'seoul' in args['location'].lower(), f'Unexpected location: {args["location"]}'

print(f'\n Tool-call 정의 완료: {tc.function.name}({args})')

finish_reason: tool_calls
content: ''
tool_calls: [ChatCompletionMessageFunctionToolCall(id='cDwfLK1oO9sEm0NDmy7As414oZspYAzr', function=Function(arguments='{"location":"Seoul"}', name='get_current_weather'), type='function')]

 Tool-call 정의 완료: get_current_weather({'location': 'Seoul'})


## 8. 도구 호출 테스트 (LangChain)

`ChatOpenAI` + `create_agent` + 실제 Python callable을 통한 동일 검증입니다. 05~09 실습에서 사용할 경로(`local_llm.get_chat_model()`)와 동일하므로, 이것이 작동하면 에이전트 실습도 작동합니다.

질문 → 도구 호출 → (더미) 도구 실행 → 결과 피드백 → 최종 답변의 한 사이클을 수행합니다.

> **LangChain**은 위의 ①~⑤ 루프, 메시지 관리, 도구 실행을 대신 해 주는 라이브러리입니다.
> `create_agent(llm, [도구들])`는 "질문 → 도구 호출 → 결과 반영 → 답변"을 자동으로 반복하는 **에이전트**를 만듭니다. (06번에서 자세히 다룹니다)

In [13]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from local_llm import get_chat_model   # 05~09 실습과 동일한 LLM 생성 함수


@tool
def get_current_weather(location: str) -> str:
    """주어진 위치의 현재 날씨를 조회합니다."""
    # 가짜 구현 — 스모크 테스트는 루프가 정상 종료되는지만 확인

    return f'{location}의 현재 날씨는 14°C, 맑음입니다.'




llm = get_chat_model(temperature=0)   # = ChatOpenAI(base_url='http://localhost:8080/v1', ...)

agent = create_agent(llm, [get_current_weather])

# 에이전트 실행: 질문 → 도구 호출 → 도구 실행 → 최종 답변
result = agent.invoke({'messages': [('user', '부산 날씨 어때?')]})
final_msg = result['messages'][-1].content

print(f'최종 답변: {final_msg}')

assert '부산' in final_msg or 'busan' in final_msg.lower() or '14' in final_msg, (
    f'최종 답변에 도구 결과가 반영되지 않음: {final_msg!r}'
)
print('\n에이전트 루프 정상 완료')


최종 답변: 부산의 현재 날씨는 14°C, 맑음입니다.

에이전트 루프 정상 완료


## 선택 실습: 후속 프레임워크 기능 전체 점검

앱 실습이 실제로 쓰는 LangChain 경로를 **실습과 똑같은 코드 패턴으로** 한 번씩 호출해 봅니다.
하나라도 FAIL이면 해당 실습이 불안정할 수 있으니, 다른 프리셋으로 바꿔 다시 점검하세요.

| 점검 | 사용하는 실습 |
|---|---|
| `bind_tools` 단일 tool 호출 (Literal 인자) | 09 LangGraph |
| `with_structured_output` (json_schema, Literal/Optional/중첩 list) | 08 Multi-agent, 09 Router |
| `create_agent` + `ToolStrategy(Union[...])`, tools 없음 | 05 Structured Output |
| `create_agent` + 일반 tool + `ToolStrategy` + `context` + 메모리 | 06 Agents |
| `create_agent` 다중 tool 루프 (검색 → 답변) | 07 RAG, 08 Multi-agent |

점검 로직은 `local_llm.run_self_test()` 에 있습니다 (터미널: `python local_llm.py`).


In [14]:
from local_llm import run_self_test

results = run_self_test()        # 5개 점검, 각 PASS/FAIL + 소요 시간 출력
assert all(r['ok'] for r in results), '일부 점검 실패 — 위 로그를 확인하고 프리셋 변경을 고려하세요.'
print('\n✅ 05~09 실습 준비 완료')


✅ 로컬 LLM 서버 연결 OK: http://localhost:8080/v1
   서버에 로드된 모델: Qwen3.5-4B-Q8_0.gguf
[PASS] bind_tools 단일 호출 (09)  (2.8s)
        tool_calls=[{'name': 'get_device_status', 'args': {'device': 'washer'}, 'id': 'ckZudLaSYQsj7wan2eRsMtOhO3xjgBQc', 'type': 'tool_call'}]
[FAIL] with_structured_output json_schema (08·09)  (1.2s)
        Report(intent='device_control', error_code='5C', key_metrics=[Metric(key='error_count', value='{error_count}')])
[PASS] ToolStrategy(Union) 파싱 (05)  (1.1s)
        ControlReq(intent='device_control', device_type='air_conditioner', action='set_temperature', target_temp_c=24)


Deserializing unregistered type local_llm.Resp from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('local_llm', 'Resp')]


[PASS] tool+ToolStrategy+context+메모리 (06)  (3.1s)
        turn1=Resp(reply="DEV-2002 세탁기의 현재 상태는 다음과 같습니다: 모델명 'Washer-A', 펌웨어 버전 '2.1.7', 마지막 오류 코드 '5C'입니다.", needs_human=False) | turn2=Resp(reply='DEV-2002 세탁기의 펌웨어 버전은 2.1.7 입니다.', needs_human=False)
[PASS] 다중 tool 루프 + 출처 인용 (07·08)  (4.3s)
        tools=['search_washing_machine', 'search_robot_vacuum'] | both=True | answer='요청하신 세탁기 5C 에러와 로봇청소기 도킹 문제 해결 방법을 알려드립니다.\n\n**세탁기 5C 에러 해결법**\n*   **배수 필터 이물질 제거:** 배수 필터에 이물질이 끼어 있을 수 있습니다. 이를 제거해야 합니다. [category:chunk0]\n*   **배수 '

결과: 4/5 PASS  (서버 모델: Qwen3.5-4B-Q8_0.gguf)


AssertionError: 일부 점검 실패 — 위 로그를 확인하고 프리셋 변경을 고려하세요.

## 10. 서버는 켜둔 채로 다음 실습으로

이 서버(`llama-server`, :8080)는 05~09 실습에서 공용으로 사용합니다.

- 04번은 별도 포트(:8090)에서 측정용 서버를 실행합니다. 두 서버를 함께 실행할 때 VRAM 여유를 확인하세요.
- 10번(SigLIP2) 전에 GPU 메모리가 부족하면 서버를 종료하세요.
- 종료: `python local_server.py stop`
- 다시 시작: `python local_server.py start 4b` 또는 `python local_server.py start 7b`
- 전체 기능 점검: `python local_llm.py`


### 내 코드에서 로컬 LLM 사용하기 (서버가 켜져 있는 동안)

**PowerShell**
```powershell
$body = @{ messages = @(@{ role = 'user'; content = '안녕?' }); temperature = 0 } | ConvertTo-Json -Depth 5
Invoke-RestMethod -Uri http://localhost:8080/v1/chat/completions -Method Post -ContentType 'application/json' -Body $body
```

**OpenAI 호환 Python 클라이언트**
```python
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8080/v1", api_key="local")
print(client.chat.completions.create(model="local", messages=[{"role": "user", "content": "안녕?"}]).choices[0].message.content)
```

**LangChain (05~09 실습 방식)**
```python
from local_llm import get_chat_model
print(get_chat_model().invoke("안녕?").content)
```

서버는 `127.0.0.1`에만 바인딩되어 현재 PC에서만 접근할 수 있습니다.


In [ ]:
import requests, torch
print('Health:', requests.get('http://127.0.0.1:8080/health', timeout=5).json())
print('GPU:', torch.cuda.get_device_name(0), torch.cuda.mem_get_info())
# Stop after finishing 05-09: from local_server import stop_server; stop_server(8080)

---

## 모든 테스트 통과 시

이 실행 후 디스크 사용량:
Windows의 llama-server 실행 파일을 로컬 프로세스로 시작합니다.
- 4B 프리셋: Qwen3.5-4B-Q8_0 GGUF ~4.5 GB + mmproj-F16 ~0.7 GB
- 7B 프리셋: Qwen2.5-7B-Instruct-Q4_K_M GGUF ~4.7 GB

검증 완료: 텍스트 채팅, 비전 입력(4B만), 도구 호출 (raw API / LangChain), 05~09 기능 전체 점검.

## 트러블슈팅

| 증상 | 해결 방법 |
|---|---|
| 0번 섹션 세 번째 명령 실패 | NVIDIA 드라이버와 `bin/llama-server.exe` 설치를 확인하세요. |
| `port already in use` | `local_server.py`와 `LLM_BASE_URL`의 포트를 동일하게 맞추고 해당 포트를 점유한 프로세스를 종료하세요. |
| Windows local process | `python local_server.py start` / `python local_server.py stop` |
| 5번 섹션 의미 없는 출력 | `--jinja` 플래그 누락; 채팅 템플릿 미적용 |
| 6번 섹션 이미지 무시 | `--mmproj` 경로 또는 `runtime/llama-server-8080.log`를 확인하세요. |
| 7번 섹션 `tool_calls` 미반환 | `--jinja` 플래그 누락 또는 모델이 스키마에 혼동 |
| 7번 섹션 잘못된 JSON args | 더 낮은 비트(Q3 이하) 양자화 사용 시 발생 가능 — Q4_K_M 이상 사용 |
| 9번 섹션 일부 FAIL | 셀을 한 번 더 실행(샘플링 변동) → 그래도 실패하면 다른 프리셋으로 서버 재기동 후 재점검 |
| 9번 섹션 `structured_response 없음` | 모델이 스키마 tool 대신 일반 텍스트로 끝낸 경우 — 7B/4B 교차 확인, 프롬프트에 "최종 응답은 스키마로" 명시 |
| `unknown argument: --chat-template-kwargs` | `setup_environment.bat`을 다시 실행해 실행 파일을 확인하세요. |
| A4000에서 OOM | 다른 GPU 프로세스나 Transformers 커널 종료, 또는 `--ctx-size`를 8192로 줄이기 |


## ✅ 정리 & 스스로 점검

**정리**
- llama-server는 GGUF 모델을 GPU에 올리고 **OpenAI 호환 API**로 제공합니다.
- 도구 호출은 "모델이 요청 → 우리 코드가 실행 → 결과를 다시 모델에게"의 반복입니다.
- 9번 점검을 통과했다면 05~09 실습 준비가 끝난 것입니다. **서버는 켜 둔 채로** 다음으로 넘어가세요.

**스스로 점검**
1. `python local_server.py stop` 후 05번을 실행하면 어떤 일이 생길까요? (해결 명령은?)
2. `--jinja`를 빼면 왜 도구 호출이 안 될까요? (힌트: 01번의 chat template)
3. 4B Q8_0과 7B Q4_K_M은 파일 크기가 비슷한데, 각각 무엇을 포기하고 무엇을 얻었을까요? (04번에서 측정)